In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from IPython.utils.capture import capture_output


def execute_selected_notebook_cells(notebook_path: Path, cell_indexes: list[int]) -> dict:
    with notebook_path.open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)

    code_cells = [cell for cell in notebook["cells"] if cell.get("cell_type") == "code"]
    namespace: dict = {}

    for cell_index in cell_indexes:
        code = "\n".join(code_cells[cell_index].get("source", []))
        with capture_output():
            exec(compile(code, str(notebook_path), "exec"), namespace)

    return namespace


workspace_root = Path.cwd().resolve()
if workspace_root.name.lower() == "notebooks":
    workspace_root = workspace_root.parent

wind_path = workspace_root / "notebooks" / "Wind_Speed.ipynb"
homes_path = workspace_root / "notebooks" / "Homes_Usage.ipynb"

wind_namespace = execute_selected_notebook_cells(wind_path, [0, 1, 2])
homes_namespace = execute_selected_notebook_cells(homes_path, [0, 1, 2, 3, 4, 5])

wind_source = wind_namespace["daily_windturbine_produced"][["date", "Windturbine_Produced"]].copy()
wind_source["date"] = pd.to_datetime(wind_source["date"]).dt.strftime("%Y-%m-%d")
wind_source = wind_source.rename(columns={"Windturbine_Produced": "Windturbine_Produced_kwh_day"})

homes_source = homes_namespace["simulation_df"][["date", "homes_shared_energy_usage_kWh"]].copy()
homes_source["date"] = pd.to_datetime(homes_source["date"]).dt.strftime("%Y-%m-%d")
homes_source = homes_source.rename(columns={"homes_shared_energy_usage_kWh": "homes_shared_energy_usage_kwh_day"})

daily_df = (
    wind_source.merge(homes_source, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

if daily_df.empty:
    raise RuntimeError("No overlapping daily rows between wind and homes outputs.")

BATTERY_CAPACITY_KWH = 2_500_000.0
battery_storage_kwh = 25_000.0
SECONDS_PER_DAY = 1.0


def show_daily_output(daily_line: dict) -> None:
    print(f"date: {daily_line['date']}")
    print(f"Windturbine_Produced_kwh_day: {daily_line['Windturbine_Produced_kwh_day']}")
    print(f"homes_shared_energy_usage_kwh_day: {daily_line['homes_shared_energy_usage_kwh_day']}")
    print(f"Battery_Percentage: {daily_line['Battery_Percentage']}")
    print(f"Energy_Lost_Day_kwh: {daily_line['Energy_Lost_Day_kwh']}")
    print(f"Battery_kWh: {daily_line['Battery_kWh']}")


def run_simulation() -> pd.DataFrame:
    simulation_rows = []
    global battery_storage_kwh

    for row in daily_df.itertuples(index=False):
        date = row.date
        wind_kwh_day = row.Windturbine_Produced_kwh_day
        homes_kwh_day = row.homes_shared_energy_usage_kwh_day

        if pd.isna(wind_kwh_day) or pd.isna(homes_kwh_day):
            continue

        wind_kwh_day = float(wind_kwh_day)
        homes_kwh_day = float(homes_kwh_day)

        if wind_kwh_day == 0.0:
            print("No_Energy_Produced")
            break

        if wind_kwh_day + battery_storage_kwh < homes_kwh_day:
            print("Energy_Ran_Out")
            break

        battery_before = battery_storage_kwh
        surplus = wind_kwh_day - homes_kwh_day

        if surplus >= 0:
            battery_storage_kwh = min(BATTERY_CAPACITY_KWH, battery_storage_kwh + surplus)
            energy_lost_day_kwh = max(0.0, battery_before + surplus - BATTERY_CAPACITY_KWH)
        else:
            battery_storage_kwh = max(0.0, battery_storage_kwh + surplus)
            energy_lost_day_kwh = 0.0

        battery_percentage = int(round((battery_storage_kwh / BATTERY_CAPACITY_KWH) * 100))

        daily_line = {
            "date": date,
            "Windturbine_Produced_kwh_day": round(wind_kwh_day, 2),
            "homes_shared_energy_usage_kwh_day": round(homes_kwh_day, 2),
            "Battery_Percentage": battery_percentage,
            "Energy_Lost_Day_kwh": round(float(energy_lost_day_kwh), 2),
            "Battery_kWh": round(battery_storage_kwh, 2),
        }
        simulation_rows.append(daily_line)

        show_daily_output(daily_line)
        time.sleep(SECONDS_PER_DAY)

    return pd.DataFrame(
        simulation_rows,
        columns=[
            "date",
            "Windturbine_Produced_kwh_day",
            "homes_shared_energy_usage_kwh_day",
            "Battery_Percentage",
            "Energy_Lost_Day_kwh",
            "Battery_kWh",
        ],
    )


distributor_simulation_df = run_simulation()
display(HTML(distributor_simulation_df.to_html(index=False)))